# ChickenStress — 02. Eksperimen Modelling (6 Konfigurasi + Early Stopping)
### Bagian 2 dari 3 (`01_image_processing.ipynb` → `02_models.ipynb` → `03_evaluate.ipynb`)

Melanjutkan dari `01_image_processing.ipynb`. Alur eksplorasi 6 konfigurasi **tetap sama seperti notebook awal**, dengan dua patch tambahan sesuai `alur_deteksi_stres_ayam.md` STEP 7:
1. **Early stopping** di setiap eksperimen.
2. **Training dua tahap (freeze backbone → fine-tuning)** + **class weighting** sebagai pengaman residual imbalance, untuk model yang memakai backbone pretrained (DINOv2, EfficientNetV2). YOLOv11 memakai mekanisme setara lewat parameter `freeze` bawaan Ultralytics.
1. YOLOv11 satu tahap (deteksi + klasifikasi) — early stopping lewat parameter `patience` bawaan Ultralytics.
2. DINOv2 satu tahap (deteksi + klasifikasi)
3. EfficientNetV2 satu tahap (deteksi + klasifikasi)
4. YOLOv11 (deteksi) + DINOv2 (klasifikasi)
5. YOLOv11 (deteksi) + EfficientNetV2 (klasifikasi)
6. YOLOv11 (deteksi) + ensemble DINOv2 & EfficientNetV2 (klasifikasi) — tidak butuh training baru

Untuk config 2/3, early stopping dipantau lewat **validation loss** (loop training sekarang juga jalan di split `valid`, bukan cuma `train`), dengan bobot terbaik (`val_loss` terendah) yang dipakai di akhir — bukan bobot epoch terakhir. Untuk config 4/5, early stopping dipantau lewat **validation accuracy (`val_acc`)** dengan bobot terbaik (`val_acc` tertinggi). Setiap model yang selesai dilatih disimpan sebagai checkpoint ke `WORK_DIR/checkpoints`, dan daftarnya disimpan ke `WORK_DIR/experiment_checkpoints.pkl` supaya `03_evaluate.ipynb` bisa memuat ulang semua model tanpa training ulang.

In [1]:
import os
import json
import random
import shutil
import copy
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch

pd.set_option("display.max_columns", None)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Reproducibility seed set to", SEED)

Reproducibility seed set to 42


## 0. Muat artefak dari `01_image_processing.ipynb` (PATCH — bridging antar notebook)

Memuat kembali semua path dan variabel (`class_to_id`, `train_df`/`valid_df`/`test_df`, `YOLO_DIR`, `DEST`, dst.) yang disimpan di akhir notebook 1, supaya notebook ini bisa langsung lanjut ke tahap modelling tanpa mengulang copy/crop/split/augmentasi.

In [2]:
# Samakan RAW_DIR dengan notebook 1 hanya untuk menghitung ulang WORK_DIR/lokasi file artefak.
RAW_DIR = r"E:\Magang DSC\Code - Image Classification Workflow\Data"
WORK_DIR = os.path.join(os.path.dirname(RAW_DIR), "Data_working")

ARTIFACT_PATH = os.path.join(WORK_DIR, "pipeline_artifacts.pkl")
with open(ARTIFACT_PATH, "rb") as f:
    artifact_state = pickle.load(f)

globals().update(artifact_state)

print("Artefak berhasil dimuat dari:", ARTIFACT_PATH)
print("YOLO_DIR:", YOLO_DIR)
print("DEST (classification split):", DEST)
print("Jumlah kelas:", len(class_to_id), class_to_id)

Artefak berhasil dimuat dari: E:\Magang DSC\Code - Image Classification Workflow\Data_working\pipeline_artifacts.pkl
YOLO_DIR: E:\Magang DSC\Code - Image Classification Workflow\Data_working\yolo_detect_dataset
DEST (classification split): E:\Magang DSC\Code - Image Classification Workflow\Data_working\dataset
Jumlah kelas: 2 {'not_stress': 0, 'stress': 1}


In [3]:
!pip install -q ultralytics transformers timm

## 7. Eksplorasi Model — 6 Konfigurasi (10 epoch, tahap eksplorasi awal)

Semua konfigurasi dilatih **25 epoch** dulu untuk eksplorasi cepat sebelum training penuh pada konfigurasi terbaik. Evaluasi akhir tiap konfigurasi memakai metrik yang sama: akurasi klasifikasi pada objek yang berhasil terdeteksi (IoU ≥ 0.5) di `yolo_detect_dataset/images/test`, supaya perbandingan adil.

In [4]:
EXPLORE_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 15  # PATCH: early stopping — stop kalau val_loss tidak membaik selama N epoch berturut-turut

# PATCH (Step 7 MD): training dua tahap -- freeze backbone (lr besar) lalu fine-tuning (lr kecil).
# EXPLORE_EPOCHS (50) dipecah jadi 2 tahap supaya total epoch eksplorasi tetap sama seperti sebelumnya.
FREEZE_EPOCHS = 25
FINETUNE_EPOCHS = EXPLORE_EPOCHS - FREEZE_EPOCHS  # 25
FREEZE_LR = 1e-3      # Tahap 1: lr lebih besar, hanya head/beberapa layer atas yang dilatih
FINETUNE_LR = 1e-5    # Tahap 2: lr lebih kecil, supaya representasi pretrained tidak rusak
UNFREEZE_LAST_N_BLOCKS = 2  # jumlah blok/lapisan terakhir backbone yang dibuka di Tahap 2

results_summary = {}

CHECKPOINT_DIR = os.path.join(WORK_DIR, "checkpoints")  # PATCH: tempat menyimpan bobot tiap eksperimen
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

### 7.1 Konfigurasi 1 — YOLOv11 satu tahap (deteksi + klasifikasi)

In [5]:
from ultralytics import YOLO

# PATCH (Step 7 MD): training dua tahap juga untuk YOLOv11, lewat parameter `freeze` bawaan Ultralytics
# (freeze=N membekukan N layer pertama arsitektur -- setara "freeze backbone").

# Tahap 1 -- freeze backbone: bekukan 10 layer pertama (backbone YOLOv11n), lr lebih besar
yolo_config1 = YOLO("yolo11n.pt")
yolo_config1.train(
    data=os.path.join(YOLO_DIR, "data.yaml"),
    epochs=FREEZE_EPOCHS,
    imgsz=480,
    batch=8,
    optimizer="AdamW",
    lr0=FREEZE_LR,
    lrf=0.01,
    box=7.5,
    cls=1.0,
    freeze=10,  # PATCH: bekukan backbone tahap 1
    patience=EARLY_STOPPING_PATIENCE,
    device="cpu",   # ganti "0" kalau ada GPU
    workers=0,
    project="ChickenStress",
    name="config1_stage1_freeze",
    exist_ok=True,
)
config1_stage1_best = os.path.join(yolo_config1.trainer.save_dir, "weights", "best.pt")
print("Tahap 1 (freeze backbone) selesai. Checkpoint:", config1_stage1_best)

# Tahap 2 -- fine-tuning: lanjut dari bobot Tahap 1, buka semua layer (freeze=0), lr lebih kecil
yolo_config1 = YOLO(config1_stage1_best)
yolo_config1.train(
    data=os.path.join(YOLO_DIR, "data.yaml"),
    epochs=FINETUNE_EPOCHS,
    imgsz=480,
    batch=8,
    optimizer="AdamW",
    lr0=FINETUNE_LR,
    lrf=0.01,
    box=7.5,
    cls=1.0,
    freeze=0,  # PATCH: buka seluruh layer utk fine-tuning
    patience=EARLY_STOPPING_PATIENCE,
    device="cpu",
    workers=0,
    project="ChickenStress",
    name="config1_stage2_finetune",
    exist_ok=True,
)

# PATCH: catat path checkpoint terbaik (dipilih otomatis oleh Ultralytics lewat early stopping)
config1_best_ckpt = os.path.join(yolo_config1.trainer.save_dir, "weights", "best.pt")
print("Checkpoint terbaik config1 (setelah fine-tuning):", config1_best_ckpt)

# Catatan: class weighting untuk YOLOv11 belum diterapkan -- Ultralytics tidak mengekspos
# parameter weight per-kelas secara langsung di API publik tanpa modifikasi loss internal.

New https://pypi.org/project/ultralytics/8.4.160 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.103  Python-3.11.9 torch-2.14.0+cpu CPU (Intel Core i7-8550U 1.80GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.0, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\Magang DSC\Code - Image Classification Workflow\Data_working\yolo_detect_dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.

### 7.2 Dataset & training loop umum untuk detektor torchvision (dipakai config 2 & 3)

In [6]:
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torchvision.transforms.functional as TF

class YoloDetectionDataset(Dataset):
    def __init__(self, images_dir, labels_dir):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.files = sorted(os.listdir(images_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_name = self.files[idx]
        img = Image.open(os.path.join(self.images_dir, file_name)).convert("RGB")
        w, h = img.size

        label_path = os.path.join(self.labels_dir, os.path.splitext(file_name)[0] + ".txt")
        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f.read().splitlines():
                    if not line.strip():
                        continue
                    cls_id, xc, yc, wn, hn = map(float, line.split())
                    xc, yc, wn, hn = xc * w, yc * h, wn * w, hn * h
                    x1, y1 = xc - wn / 2, yc - hn / 2
                    x2, y2 = xc + wn / 2, yc + hn / 2
                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls_id) + 1)  # +1 karena 0 = background

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4)),
            "labels": torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64),
        }
        return TF.to_tensor(img), target

def collate_fn(batch):
    return tuple(zip(*batch))

train_detect_ds = YoloDetectionDataset(
    os.path.join(YOLO_DIR, "images", "train"),
    os.path.join(YOLO_DIR, "labels", "train"),
)
train_detect_loader = DataLoader(train_detect_ds, batch_size=4, shuffle=True, collate_fn=collate_fn)

# PATCH: loader validasi (dipakai untuk early stopping, bukan untuk training)
valid_detect_ds = YoloDetectionDataset(
    os.path.join(YOLO_DIR, "images", "valid"),
    os.path.join(YOLO_DIR, "labels", "valid"),
)
valid_detect_loader = DataLoader(valid_detect_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)

def freeze_detector_backbone(model):
    """PATCH (Step 7 MD) Tahap 1: bekukan backbone pretrained (DINOv2/EfficientNetV2) di dalam
    FasterRCNN. RPN + ROI heads (yang masih random-init) tetap trainable."""
    for p in model.backbone.parameters():
        p.requires_grad = False
    for name, module in model.named_children():
        if name != "backbone":
            for p in module.parameters():
                p.requires_grad = True


def unfreeze_detector_backbone_last_blocks(model, n=2):
    """PATCH (Step 7 MD) Tahap 2: buka n blok/lapisan terakhir backbone untuk fine-tuning."""
    backbone = model.backbone
    if hasattr(backbone, "dinov2"):  # Dinov2Backbone
        layers = backbone.dinov2.encoder.layer
        for layer in layers[-n:]:
            for p in layer.parameters():
                p.requires_grad = True
    elif hasattr(backbone, "features"):  # EfficientNetV2Backbone
        blocks = list(backbone.features)
        for block in blocks[-n:]:
            for p in block.parameters():
                p.requires_grad = True
    else:
        raise ValueError("Tipe backbone tidak dikenali untuk unfreeze_detector_backbone_last_blocks().")


def _run_detector_epochs(model, train_loader, valid_loader, epochs, lr, device, patience,
                          phase_name, best_val_loss, best_state, epochs_no_improve=0):
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for images, targets in train_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
        train_loss = epoch_loss / len(train_loader)

        # PATCH: FasterRCNN hanya mengembalikan loss (bukan prediksi) saat model.train() + targets diberikan,
        # jadi validasi tetap pakai model.train() tapi dibungkus torch.no_grad() supaya bobot tidak ikut ter-update.
        val_loss = 0.0
        with torch.no_grad():
            for images, targets in valid_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                loss_dict = model(images, targets)
                val_loss += sum(loss_dict.values()).item()
        val_loss /= len(valid_loader)

        print(f"[{phase_name}] Epoch {epoch+1}/{epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[Early stopping - {phase_name}] val_loss tidak membaik selama {patience} epoch, berhenti di epoch {epoch+1}.")
                break

    return best_val_loss, best_state


def train_torchvision_detector_two_stage(
    model, train_loader, valid_loader,
    freeze_epochs=FREEZE_EPOCHS, finetune_epochs=FINETUNE_EPOCHS,
    freeze_lr=FREEZE_LR, finetune_lr=FINETUNE_LR, unfreeze_last_n=UNFREEZE_LAST_N_BLOCKS,
    device="cpu", patience=EARLY_STOPPING_PATIENCE, checkpoint_path=None,
):
    """PATCH (Step 7 MD): training dua tahap untuk detektor berbasis backbone pretrained.
    Tahap 1 -- freeze backbone: hanya RPN + ROI heads dilatih, lr besar.
    Tahap 2 -- fine-tuning: buka `unfreeze_last_n` blok terakhir backbone, lr kecil.
    Bobot akhir = bobot dengan val_loss terbaik di antara kedua tahap.
    Catatan: class weighting belum diterapkan di sini -- FasterRCNN tidak mengekspos parameter
    weight per-kelas secara langsung tanpa memodifikasi roi_heads.fastrcnn_loss."""
    model.to(device)
    best_val_loss, best_state = float("inf"), None

    freeze_detector_backbone(model)
    print(f"=== Tahap 1: Freeze backbone, hanya RPN+ROI heads dilatih (lr={freeze_lr}) ===")
    best_val_loss, best_state = _run_detector_epochs(
        model, train_loader, valid_loader, freeze_epochs, freeze_lr, device, patience,
        "Tahap1-FreezeBackbone", best_val_loss, best_state,
    )

    unfreeze_detector_backbone_last_blocks(model, n=unfreeze_last_n)
    print(f"=== Tahap 2: Fine-tuning, {unfreeze_last_n} blok terakhir backbone dibuka (lr={finetune_lr}) ===")
    best_val_loss, best_state = _run_detector_epochs(
        model, train_loader, valid_loader, finetune_epochs, finetune_lr, device, patience,
        "Tahap2-FineTuning", best_val_loss, best_state,
    )

    if best_state is not None:
        model.load_state_dict(best_state)

    if checkpoint_path:
        torch.save(model.state_dict(), checkpoint_path)
        print("Model tersimpan ke:", checkpoint_path)

    return model

### 7.3 Konfigurasi 2 — DINOv2 satu tahap (deteksi + klasifikasi)

DINOv2 dipasang sebagai backbone `FasterRCNN` (token patch ViT direshape jadi feature map).

In [7]:
from transformers import Dinov2Model
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign

class Dinov2Backbone(nn.Module):
    """Bungkus DINOv2 (ViT) supaya kompatibel sebagai backbone FasterRCNN,
    dengan reshape token patch kembali menjadi feature map spasial."""

    def __init__(self, model_name="facebook/dinov2-small", patch_size=14):
        super().__init__()
        self.dinov2 = Dinov2Model.from_pretrained(model_name)
        self.patch_size = patch_size
        self.out_channels = self.dinov2.config.hidden_size

    def forward(self, x):
        h, w = x.shape[-2] // self.patch_size, x.shape[-1] // self.patch_size
        tokens = self.dinov2(x).last_hidden_state[:, 1:, :]  # buang token CLS
        b, n, c = tokens.shape
        feat_map = tokens.permute(0, 2, 1).reshape(b, c, h, w)
        return {"0": feat_map}

def build_dinov2_detector(num_classes):
    backbone = Dinov2Backbone()
    anchor_generator = AnchorGenerator(
        sizes=((32, 64, 128, 256, 512),),
        aspect_ratios=((0.5, 1.0, 2.0),),
    )
    roi_pooler = MultiScaleRoIAlign(featmap_names=["0"], output_size=7, sampling_ratio=2)
    model = FasterRCNN(
        backbone,
        num_classes=num_classes + 1,  # +1 untuk background
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,
    )
    return model

config2_model = build_dinov2_detector(num_classes=len(class_to_id))
config2_ckpt = os.path.join(CHECKPOINT_DIR, "config2_dinov2_detector.pt")  # PATCH
config2_model = train_torchvision_detector_two_stage(
    config2_model, train_detect_loader, valid_detect_loader,
    patience=EARLY_STOPPING_PATIENCE, checkpoint_path=config2_ckpt,
)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

=== Tahap 1: Freeze backbone, hanya RPN+ROI heads dilatih (lr=0.001) ===
[Tahap1-FreezeBackbone] Epoch 1/25 - train_loss: 3.2565 - val_loss: 1.6821
[Tahap1-FreezeBackbone] Epoch 2/25 - train_loss: 1.5365 - val_loss: 1.0189
[Tahap1-FreezeBackbone] Epoch 3/25 - train_loss: 0.9887 - val_loss: 0.8105
[Tahap1-FreezeBackbone] Epoch 4/25 - train_loss: 0.9366 - val_loss: 0.8368
[Tahap1-FreezeBackbone] Epoch 5/25 - train_loss: 0.8334 - val_loss: 0.6878
[Tahap1-FreezeBackbone] Epoch 6/25 - train_loss: 0.7315 - val_loss: 0.7525
[Tahap1-FreezeBackbone] Epoch 7/25 - train_loss: 0.7485 - val_loss: 0.7386
[Tahap1-FreezeBackbone] Epoch 8/25 - train_loss: 0.7387 - val_loss: 0.7866
[Tahap1-FreezeBackbone] Epoch 9/25 - train_loss: 0.7369 - val_loss: 0.7785
[Tahap1-FreezeBackbone] Epoch 10/25 - train_loss: 0.6166 - val_loss: 0.7653
[Tahap1-FreezeBackbone] Epoch 11/25 - train_loss: 0.6118 - val_loss: 0.7434
[Tahap1-FreezeBackbone] Epoch 12/25 - train_loss: 0.6136 - val_loss: 0.7999
[Tahap1-FreezeBackbone] 

### 7.4 Konfigurasi 3 — EfficientNetV2 satu tahap (deteksi + klasifikasi)

In [8]:
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

class EfficientNetV2Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        base = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
        self.features = base.features
        self.out_channels = 1280  # channel terakhir EfficientNetV2-S

    def forward(self, x):
        feat_map = self.features(x)
        return {"0": feat_map}

def build_effnetv2_detector(num_classes):
    backbone = EfficientNetV2Backbone()
    anchor_generator = AnchorGenerator(
        sizes=((32, 64, 128, 256, 512),),
        aspect_ratios=((0.5, 1.0, 2.0),),
    )
    roi_pooler = MultiScaleRoIAlign(featmap_names=["0"], output_size=7, sampling_ratio=2)
    model = FasterRCNN(
        backbone,
        num_classes=num_classes + 1,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,
    )
    return model

config3_model = build_effnetv2_detector(num_classes=len(class_to_id))
config3_ckpt = os.path.join(CHECKPOINT_DIR, "config3_effnetv2_detector.pt")  # PATCH
config3_model = train_torchvision_detector_two_stage(
    config3_model, train_detect_loader, valid_detect_loader,
    patience=EARLY_STOPPING_PATIENCE, checkpoint_path=config3_ckpt,
)

=== Tahap 1: Freeze backbone, hanya RPN+ROI heads dilatih (lr=0.001) ===
[Tahap1-FreezeBackbone] Epoch 1/25 - train_loss: 2.6969 - val_loss: 1.2544
[Tahap1-FreezeBackbone] Epoch 2/25 - train_loss: 1.3853 - val_loss: 0.9111
[Tahap1-FreezeBackbone] Epoch 3/25 - train_loss: 0.8132 - val_loss: 0.6785
[Tahap1-FreezeBackbone] Epoch 4/25 - train_loss: 0.7359 - val_loss: 0.6045
[Tahap1-FreezeBackbone] Epoch 5/25 - train_loss: 0.6167 - val_loss: 0.4199
[Tahap1-FreezeBackbone] Epoch 6/25 - train_loss: 0.5559 - val_loss: 0.5888
[Tahap1-FreezeBackbone] Epoch 7/25 - train_loss: 0.5287 - val_loss: 0.5094
[Tahap1-FreezeBackbone] Epoch 8/25 - train_loss: 0.3847 - val_loss: 0.5183
[Tahap1-FreezeBackbone] Epoch 9/25 - train_loss: 0.5016 - val_loss: 0.5726
[Tahap1-FreezeBackbone] Epoch 10/25 - train_loss: 0.3922 - val_loss: 0.4035
[Tahap1-FreezeBackbone] Epoch 11/25 - train_loss: 0.3448 - val_loss: 0.6446
[Tahap1-FreezeBackbone] Epoch 12/25 - train_loss: 0.3248 - val_loss: 0.4011
[Tahap1-FreezeBackbone] 

### 7.5 Classifier training utility untuk pipeline dua-tahap (config 4, 5, 6)

Classifier dilatih di atas crop **train yang sudah dibalancing** (`DEST/train`), sesuai patch nomor 2.

In [9]:
class CropClassificationDataset(Dataset):
    def __init__(self, root_dir, class_to_id, transform=None):
        self.samples = []
        self.transform = transform
        for cls, idx in class_to_id.items():
            folder = os.path.join(root_dir, cls)
            if not os.path.isdir(folder):
                continue
            for fname in os.listdir(folder):
                self.samples.append((os.path.join(folder, fname), idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

from torchvision import transforms as T

classifier_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

train_cls_ds = CropClassificationDataset(os.path.join(DEST, "train"), class_to_id, classifier_transform)
train_cls_loader = DataLoader(train_cls_ds, batch_size=16, shuffle=True)

# PATCH: loader validasi (dipakai untuk early stopping, bukan untuk training)
valid_cls_ds = CropClassificationDataset(os.path.join(DEST, "valid"), class_to_id, classifier_transform)
valid_cls_loader = DataLoader(valid_cls_ds, batch_size=16, shuffle=False)

from collections import Counter

def freeze_backbone(model):
    """PATCH (Step 7 MD) Tahap 1: bekukan seluruh parameter backbone pretrained.
    Classification head tetap trainable. Mendukung Dinov2Classifier (.backbone/.head)
    dan EfficientNetV2Classifier (.model.features/.model.classifier)."""
    if hasattr(model, "backbone") and hasattr(model, "head"):  # Dinov2Classifier
        for p in model.backbone.parameters():
            p.requires_grad = False
        for p in model.head.parameters():
            p.requires_grad = True
    elif hasattr(model, "model") and hasattr(model.model, "features"):  # EfficientNetV2Classifier
        for p in model.model.features.parameters():
            p.requires_grad = False
        for p in model.model.classifier.parameters():
            p.requires_grad = True
    else:
        raise ValueError("Tipe model tidak dikenali untuk freeze_backbone().")


def unfreeze_last_blocks(model, n=2):
    """PATCH (Step 7 MD) Tahap 2: buka n blok/lapisan terakhir backbone untuk fine-tuning."""
    if hasattr(model, "backbone") and hasattr(model, "head"):  # Dinov2Classifier
        layers = model.backbone.encoder.layer
        for layer in layers[-n:]:
            for p in layer.parameters():
                p.requires_grad = True
    elif hasattr(model, "model") and hasattr(model.model, "features"):  # EfficientNetV2Classifier
        blocks = list(model.model.features)
        for block in blocks[-n:]:
            for p in block.parameters():
                p.requires_grad = True
    else:
        raise ValueError("Tipe model tidak dikenali untuk unfreeze_last_blocks().")


def compute_class_weights(dataset, num_classes, device="cpu"):
    """PATCH (Step 7 MD): class weighting sebagai pengaman residual imbalance.
    Meski dataset sudah dibalancing di Step 2 (01_image_processing.ipynb), distribusi
    bisa sedikit bergeser setelah split train/valid/test -- weight ini menjaga supaya
    kelas yang sedikit lebih sedikit di split train tetap diberi bobot loss lebih besar."""
    counts = Counter(label for _, label in dataset.samples)
    total = sum(counts.values())
    weights = torch.tensor(
        [total / (num_classes * counts.get(i, 1)) for i in range(num_classes)],
        dtype=torch.float32,
    )
    return weights.to(device)


def _run_classifier_epochs(model, train_loader, valid_loader, epochs, lr, criterion, device, patience,
                            phase_name, best_val_acc, best_state, epochs_no_improve=0):
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    for epoch in range(epochs):
        model.train()
        epoch_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        train_loss = epoch_loss / len(train_loader)
        train_acc = correct / total

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in valid_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)
        val_loss /= len(valid_loader)
        val_acc = val_correct / val_total

        print(f"[{phase_name}] Epoch {epoch+1}/{epochs} - train_loss: {train_loss:.4f} - train_acc: {train_acc:.4f} "
              f"- val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")

        if val_acc > best_val_acc + 1e-4:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[Early stopping - {phase_name}] val_acc tidak membaik selama {patience} epoch, berhenti di epoch {epoch+1}.")
                break

    return best_val_acc, best_state


def train_classifier_two_stage(
    model, train_loader, valid_loader,
    freeze_epochs=FREEZE_EPOCHS, finetune_epochs=FINETUNE_EPOCHS,
    freeze_lr=FREEZE_LR, finetune_lr=FINETUNE_LR, unfreeze_last_n=UNFREEZE_LAST_N_BLOCKS,
    class_weights=None, device="cpu", patience=EARLY_STOPPING_PATIENCE, checkpoint_path=None,
):
    """PATCH (Step 7 MD): training classifier dua tahap.
    Tahap 1 -- freeze backbone: hanya classification head dilatih, lr besar (mis. 1e-3).
    Tahap 2 -- fine-tuning: buka `unfreeze_last_n` blok terakhir backbone, lr kecil (mis. 1e-5)
    supaya representasi pretrained tidak rusak.
    class_weights: tensor pengaman residual imbalance untuk CrossEntropyLoss.
    Bobot akhir = bobot dengan val_acc terbaik di antara kedua tahap."""
    model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_val_acc, best_state = float("-inf"), None

    freeze_backbone(model)
    print(f"=== Tahap 1: Freeze backbone, hanya classification head dilatih (lr={freeze_lr}) ===")
    best_val_acc, best_state = _run_classifier_epochs(
        model, train_loader, valid_loader, freeze_epochs, freeze_lr, criterion, device, patience,
        "Tahap1-FreezeBackbone", best_val_acc, best_state,
    )

    unfreeze_last_blocks(model, n=unfreeze_last_n)
    print(f"=== Tahap 2: Fine-tuning, {unfreeze_last_n} blok terakhir backbone dibuka (lr={finetune_lr}) ===")
    best_val_acc, best_state = _run_classifier_epochs(
        model, train_loader, valid_loader, finetune_epochs, finetune_lr, criterion, device, patience,
        "Tahap2-FineTuning", best_val_acc, best_state,
    )

    if best_state is not None:
        model.load_state_dict(best_state)

    if checkpoint_path:
        torch.save(model.state_dict(), checkpoint_path)
        print("Model tersimpan ke:", checkpoint_path)

    return model

# PATCH (Step 7 MD): class weights dihitung sekali dari distribusi train_cls_ds, dipakai
# ulang oleh config 4 & 5 (keduanya classifier dua-tahap).
class_weights = compute_class_weights(train_cls_ds, num_classes=len(class_to_id))
print("Class weights (pengaman residual imbalance):", dict(zip(class_to_id.keys(), class_weights.tolist())))

Class weights (pengaman residual imbalance): {'not_stress': 1.0, 'stress': 1.0}


### 7.6 Konfigurasi 4 — YOLOv11 (deteksi) + DINOv2 (klasifikasi)

In [10]:
class Dinov2Classifier(nn.Module):
    def __init__(self, num_classes, model_name="facebook/dinov2-small"):
        super().__init__()
        self.backbone = Dinov2Model.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x):
        feat = self.backbone(x).last_hidden_state[:, 0, :]  # token CLS
        return self.head(feat)

config4_classifier = Dinov2Classifier(num_classes=len(class_to_id))
config4_ckpt = os.path.join(CHECKPOINT_DIR, "config4_dinov2_classifier.pt")  # PATCH
config4_classifier = train_classifier_two_stage(
    config4_classifier, train_cls_loader, valid_cls_loader,
    class_weights=class_weights, patience=EARLY_STOPPING_PATIENCE, checkpoint_path=config4_ckpt,
)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

=== Tahap 1: Freeze backbone, hanya classification head dilatih (lr=0.001) ===
[Tahap1-FreezeBackbone] Epoch 1/25 - train_loss: 1.1130 - train_acc: 0.5000 - val_loss: 0.8657 - val_acc: 0.3333
[Tahap1-FreezeBackbone] Epoch 2/25 - train_loss: 0.6611 - train_acc: 0.6296 - val_loss: 0.9891 - val_acc: 0.4167
[Tahap1-FreezeBackbone] Epoch 3/25 - train_loss: 0.4195 - train_acc: 0.7407 - val_loss: 0.5303 - val_acc: 0.7500
[Tahap1-FreezeBackbone] Epoch 4/25 - train_loss: 0.2332 - train_acc: 0.9444 - val_loss: 0.3894 - val_acc: 0.8333
[Tahap1-FreezeBackbone] Epoch 5/25 - train_loss: 0.1847 - train_acc: 0.9074 - val_loss: 0.3227 - val_acc: 0.8333
[Tahap1-FreezeBackbone] Epoch 6/25 - train_loss: 0.1319 - train_acc: 0.9815 - val_loss: 0.2569 - val_acc: 0.9167
[Tahap1-FreezeBackbone] Epoch 7/25 - train_loss: 0.1025 - train_acc: 1.0000 - val_loss: 0.2562 - val_acc: 1.0000
[Tahap1-FreezeBackbone] Epoch 8/25 - train_loss: 0.0760 - train_acc: 0.9815 - val_loss: 0.2832 - val_acc: 1.0000
[Tahap1-FreezeBac

### 7.7 Konfigurasi 5 — YOLOv11 (deteksi) + EfficientNetV2 (klasifikasi)

In [11]:
class EfficientNetV2Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
        in_features = base.classifier[1].in_features
        base.classifier[1] = nn.Linear(in_features, num_classes)
        self.model = base

    def forward(self, x):
        return self.model(x)

config5_classifier = EfficientNetV2Classifier(num_classes=len(class_to_id))
config5_ckpt = os.path.join(CHECKPOINT_DIR, "config5_effnetv2_classifier.pt")  # PATCH
config5_classifier = train_classifier_two_stage(
    config5_classifier, train_cls_loader, valid_cls_loader,
    class_weights=class_weights, patience=EARLY_STOPPING_PATIENCE, checkpoint_path=config5_ckpt,
)

=== Tahap 1: Freeze backbone, hanya classification head dilatih (lr=0.001) ===
[Tahap1-FreezeBackbone] Epoch 1/25 - train_loss: 0.6656 - train_acc: 0.5926 - val_loss: 0.6948 - val_acc: 0.6667
[Tahap1-FreezeBackbone] Epoch 2/25 - train_loss: 0.5505 - train_acc: 0.8333 - val_loss: 0.6952 - val_acc: 0.5000
[Tahap1-FreezeBackbone] Epoch 3/25 - train_loss: 0.5216 - train_acc: 0.8704 - val_loss: 0.6485 - val_acc: 0.6667
[Tahap1-FreezeBackbone] Epoch 4/25 - train_loss: 0.4772 - train_acc: 0.8704 - val_loss: 0.6038 - val_acc: 0.6667
[Tahap1-FreezeBackbone] Epoch 5/25 - train_loss: 0.3744 - train_acc: 0.9444 - val_loss: 0.5572 - val_acc: 0.8333
[Tahap1-FreezeBackbone] Epoch 6/25 - train_loss: 0.3464 - train_acc: 1.0000 - val_loss: 0.5166 - val_acc: 0.8333
[Tahap1-FreezeBackbone] Epoch 7/25 - train_loss: 0.3188 - train_acc: 0.9815 - val_loss: 0.5277 - val_acc: 0.8333
[Tahap1-FreezeBackbone] Epoch 8/25 - train_loss: 0.3306 - train_acc: 0.9630 - val_loss: 0.5431 - val_acc: 0.6667
[Tahap1-FreezeBac

### 7.8 Konfigurasi 6 — YOLOv11 (deteksi) + Ensemble(DINOv2, EfficientNetV2)

Tidak perlu training baru — konfigurasi ini menggunakan ulang detektor (config 1) dan kedua classifier yang sudah dilatih (config 4 & 5), digabung lewat rata-rata softmax saat inferensi.

## 9. Simpan daftar checkpoint eksperimen (PATCH — bridging antar notebook)

Menyimpan lokasi checkpoint tiap eksperimen (bukan modelnya sendiri — model di-load ulang dari file ini) ke `WORK_DIR/experiment_checkpoints.pkl`, supaya `03_evaluate.ipynb` bisa memuat ulang keenam model tanpa training ulang.

In [12]:
experiment_checkpoints = {
    "1. YOLOv11 (satu tahap)": {"type": "yolo", "checkpoint": config1_best_ckpt},
    "2. DINOv2 (satu tahap)": {"type": "torchvision_detector", "backbone": "dinov2", "checkpoint": config2_ckpt},
    "3. EfficientNetV2 (satu tahap)": {"type": "torchvision_detector", "backbone": "effnetv2", "checkpoint": config3_ckpt},
    "4. YOLOv11 + DINOv2": {"type": "two_stage", "classifier": "dinov2", "checkpoint": config4_ckpt, "detector": "1. YOLOv11 (satu tahap)"},
    "5. YOLOv11 + EfficientNetV2": {"type": "two_stage", "classifier": "effnetv2", "checkpoint": config5_ckpt, "detector": "1. YOLOv11 (satu tahap)"},
    "6. YOLOv11 + Ensemble(DINOv2, EfficientNetV2)": {"type": "two_stage_ensemble", "checkpoints": [config4_ckpt, config5_ckpt], "detector": "1. YOLOv11 (satu tahap)"},
}

# PATCH (fix): simpan EXPLORE_EPOCHS & EARLY_STOPPING_PATIENCE juga, dipakai lagi di 03_evaluate.ipynb
# PATCH (Step 7 MD): simpan juga metadata training dua tahap (freeze->fine-tune) & class weights
experiment_checkpoints["_meta"] = {
    "EXPLORE_EPOCHS": EXPLORE_EPOCHS,
    "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
    "FREEZE_EPOCHS": FREEZE_EPOCHS,
    "FINETUNE_EPOCHS": FINETUNE_EPOCHS,
    "FREEZE_LR": FREEZE_LR,
    "FINETUNE_LR": FINETUNE_LR,
    "UNFREEZE_LAST_N_BLOCKS": UNFREEZE_LAST_N_BLOCKS,
    "class_weights": class_weights.tolist(),
}

EXPERIMENT_REGISTRY_PATH = os.path.join(WORK_DIR, "experiment_checkpoints.pkl")
with open(EXPERIMENT_REGISTRY_PATH, "wb") as f:
    pickle.dump(experiment_checkpoints, f)

print("Daftar checkpoint eksperimen disimpan ke:", EXPERIMENT_REGISTRY_PATH)
for name, info in experiment_checkpoints.items():
    if name == "_meta":
        continue
    print(" -", name, "->", info)

Daftar checkpoint eksperimen disimpan ke: E:\Magang DSC\Code - Image Classification Workflow\Data_working\experiment_checkpoints.pkl
 - 1. YOLOv11 (satu tahap) -> {'type': 'yolo', 'checkpoint': 'E:\\Magang DSC\\Code - Image Classification Workflow\\runs\\detect\\ChickenStress\\config1_stage2_finetune\\weights\\best.pt'}
 - 2. DINOv2 (satu tahap) -> {'type': 'torchvision_detector', 'backbone': 'dinov2', 'checkpoint': 'E:\\Magang DSC\\Code - Image Classification Workflow\\Data_working\\checkpoints\\config2_dinov2_detector.pt'}
 - 3. EfficientNetV2 (satu tahap) -> {'type': 'torchvision_detector', 'backbone': 'effnetv2', 'checkpoint': 'E:\\Magang DSC\\Code - Image Classification Workflow\\Data_working\\checkpoints\\config3_effnetv2_detector.pt'}
 - 4. YOLOv11 + DINOv2 -> {'type': 'two_stage', 'classifier': 'dinov2', 'checkpoint': 'E:\\Magang DSC\\Code - Image Classification Workflow\\Data_working\\checkpoints\\config4_dinov2_classifier.pt', 'detector': '1. YOLOv11 (satu tahap)'}
 - 5. YOLO